# Phase 2.5 — BERT Embeddings as Feature Extraction
## Drug Review Dataset — Transfer Learning

**TAB Group Project**

---

### Overview

This notebook covers **Phase 2.5 — BERT Embeddings as Feature Extraction**.

It sits between:
- **Phase 2** (R Markdown) — traditional feature engineering (AFINN, TF-IDF, etc.)
- **Phase 3** — modelling (classifier training and evaluation)

We use a pre-trained **BERT** model to convert each drug review into a **768-dimensional embedding vector** that captures the deep semantic meaning of the text. These embeddings are added as features to our dataset and passed to Phase 3 alongside the features from Phase 2.

### Why this is Transfer Learning

BERT was pre-trained on **3.3 billion words** from Wikipedia and BooksCorpus. By using its embeddings, we transfer that rich language understanding into our drug review classification tasks — without needing to train a model from scratch.

| Approach | What it means |
|---|---|
| **Feature extraction (this notebook)** | BERT weights are frozen — we just extract embeddings |
| Fine-tuning | BERT weights are updated on our specific task |

### Pipeline

```
phase0_phase2.Rmd  →  drug_reviews_phase2.csv
                              ↓
                   phase2_5_bert.ipynb  (this notebook)
                              ↓
                   drug_reviews_phase2_5.csv
                              ↓
                          Phase 3
```

---
## Step 1 — Install Required Libraries

In [ ]:
# Run this cell once to install required packages
# You can skip this if they are already installed
import subprocess
subprocess.run(['pip', 'install', 'transformers', 'torch', 'numpy', 'pandas', 'tqdm'], 
               check=True)

---
## Step 2 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

# Check if GPU is available (much faster than CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

if device == 'cpu':
    print('\nNote: Running on CPU. Consider using Google Colab with GPU for faster processing.')
    print('Estimated time on CPU: 3-4 hours for full dataset')
    print('Estimated time on GPU: ~20 minutes for full dataset')

---
## Step 3 — Load Dataset from Phase 2

In [ ]:
# Load the enriched dataset from Phase 2
df = pd.read_csv('drug_reviews_phase2.csv')

print(f'Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')

In [ ]:
# Preview the data
df[['uniqueID', 'drugName', 'review_clean', 'rating', 'useful_binary']].head(5)

---
## Step 4 — Optional: Use a Sample (Recommended for CPU)

Running BERT on 200,000 reviews on CPU takes 3-4 hours.
If you are on CPU, we recommend using a stratified sample of 50,000 reviews.

**Set `USE_SAMPLE = False` if you have a GPU and want to process all reviews.**

In [ ]:
USE_SAMPLE = True    # Set to False if you have GPU
SAMPLE_SIZE = 50000  # Number of reviews to use if sampling
RANDOM_SEED = 42

if USE_SAMPLE:
    # Stratified sample to keep class balance of useful_binary
    df = df.groupby('useful_binary', group_keys=False).apply(
        lambda x: x.sample(frac=SAMPLE_SIZE/len(df), random_state=RANDOM_SEED)
    ).reset_index(drop=True)
    print(f'Using stratified sample: {len(df)} reviews')
    print(f'Class balance:')
    print(df['useful_binary'].value_counts())
else:
    print(f'Using full dataset: {len(df)} reviews')

---
## Step 5 — Load Pre-trained BERT Model

We use `bert-base-uncased` from HuggingFace:
- **bert-base** — 12 transformer layers, 768 hidden dimensions, 110M parameters
- **uncased** — treats uppercase and lowercase the same (matches our lowercased `review_clean`)
- **Pre-trained on** — Wikipedia (2.5B words) + BooksCorpus (800M words)

The model is set to **evaluation mode** (`model.eval()`) because we are NOT training it.
We are using it purely as a frozen feature extractor — this is what makes it **transfer learning**.

In [ ]:
MODEL_NAME = 'bert-base-uncased'

print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f'Loading BERT model: {MODEL_NAME} ...')
model = AutoModel.from_pretrained(MODEL_NAME)

# Move model to GPU if available
model = model.to(device)

# Set to evaluation mode - weights are FROZEN, no training happening
model.eval()

print(f'\nBERT model loaded successfully')
print(f'Model is on: {next(model.parameters()).device}')
print(f'Embedding dimensions: 768')

---
## Step 6 — Understand BERT Tokenisation

Before generating embeddings for all reviews, let us look at what BERT tokenisation does to a single review.

In [ ]:
# Show what BERT tokenisation looks like on one example
example = 'this drug helped my anxiety but caused some drowsiness'

tokens = tokenizer.tokenize(example)
print(f'Original text: "{example}"')
print(f'\nBERT tokens: {tokens}')
print(f'\nNote: BERT adds [CLS] at the start and [SEP] at the end')
print(f'      [CLS] embedding = the sentence-level representation we extract')

# Show with special tokens
encoded = tokenizer.encode(example)
print(f'\nWith special tokens: {tokenizer.convert_ids_to_tokens(encoded)}')

---
## Step 7 — Generate BERT Embeddings

We process reviews in **batches** for efficiency. For each review:
1. Tokenise into BERT subword tokens
2. Add `[CLS]` and `[SEP]` special tokens
3. Pass through frozen BERT
4. Extract the `[CLS]` token embedding (768 values) as the review representation

**Why `[CLS]` token?**
BERT is designed so that the `[CLS]` token at position 0 aggregates information
from the entire sequence. It is the standard choice for sentence-level tasks
like classification.

In [ ]:
def get_bert_embeddings_batch(texts, batch_size=32, max_length=128):
    """
    Generate BERT embeddings for a list of texts.
    
    Parameters:
        texts      : list of review strings
        batch_size : number of reviews to process at once
                     (reduce to 16 if you run out of memory)
        max_length : maximum token length per review
                     (128 covers most reviews and is faster than 512)
    
    Returns:
        embeddings : numpy array of shape (n_reviews, 768)
    """
    all_embeddings = []
    
    # Process in batches with progress bar
    for i in tqdm(range(0, len(texts), batch_size), desc='Generating embeddings'):
        
        batch = texts[i : i + batch_size]
        
        # Tokenise the batch
        inputs = tokenizer(
            batch,
            return_tensors = 'pt',
            truncation     = True,
            padding        = 'max_length',
            max_length     = max_length
        )
        
        # Move inputs to same device as model
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Forward pass — no gradient computation (not training)
        with torch.no_grad():
            outputs = model(**inputs)
        
        # Extract [CLS] token embedding: shape (batch_size, 768)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        
        # Move to CPU and convert to numpy
        all_embeddings.append(cls_embeddings.cpu().numpy())
    
    # Stack all batches: shape (n_reviews, 768)
    return np.vstack(all_embeddings)


# Prepare reviews — use review_clean from Phase 0 (lowercased, HTML-stripped)
reviews = df['review_clean'].fillna('').tolist()

print(f'Generating embeddings for {len(reviews)} reviews...')
print(f'Batch size: 32 | Max token length: 128')
print(f'Device: {device}\n')

# Generate embeddings
embeddings_matrix = get_bert_embeddings_batch(reviews, batch_size=32, max_length=128)

print(f'\nEmbeddings generated successfully')
print(f'Embeddings matrix shape: {embeddings_matrix.shape}')
print(f'Expected: ({len(reviews)}, 768)')

---
## Step 8 — Verify Embeddings

In [ ]:
# Basic checks
print('=== Embedding Verification ===')
print(f'Shape: {embeddings_matrix.shape}')
print(f'Data type: {embeddings_matrix.dtype}')
print(f'Min value: {embeddings_matrix.min():.4f}')
print(f'Max value: {embeddings_matrix.max():.4f}')
print(f'Mean value: {embeddings_matrix.mean():.4f}')
print(f'Any NaN: {np.isnan(embeddings_matrix).any()}')
print(f'\nFirst review embedding (first 10 values):')
print(embeddings_matrix[0, :10].round(4))

In [ ]:
# Sanity check — similar reviews should have higher cosine similarity
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Show the actual reviews being compared
print('Review 1:', reviews[0][:80], '...')
print('Review 2:', reviews[1][:80], '...')
print('Review 100:', reviews[99][:80], '...')

sim_1_2   = cosine_similarity(embeddings_matrix[0], embeddings_matrix[1])
sim_1_100 = cosine_similarity(embeddings_matrix[0], embeddings_matrix[99])

print(f'\nCosine similarity between review 1 and review 2:   {sim_1_2:.4f}')
print(f'Cosine similarity between review 1 and review 100: {sim_1_100:.4f}')
print('\nBERT embeddings range from -1 to 1 in cosine similarity')
print('Higher = more semantically similar')

---
## Step 9 — Save Embeddings

We save in two formats:
- `bert_embeddings.npy` — fast binary format for reloading later
- `drug_reviews_phase2_5.csv` — full dataset with all features + BERT embeddings for Phase 3

In [ ]:
# Save raw embeddings matrix (fast to reload)
np.save('bert_embeddings.npy', embeddings_matrix)
print('Saved: bert_embeddings.npy')

# Create dataframe of embeddings with column names bert_1 ... bert_768
embedding_cols = [f'bert_{i+1}' for i in range(768)]
embeddings_df  = pd.DataFrame(embeddings_matrix, columns=embedding_cols)

# Combine with original Phase 2 dataframe
df_final = pd.concat(
    [df.reset_index(drop=True), embeddings_df.reset_index(drop=True)],
    axis=1
)

# Save final enriched dataset
df_final.to_csv('drug_reviews_phase2_5.csv', index=False)

print(f'Saved: drug_reviews_phase2_5.csv')
print(f'Final shape: {df_final.shape}')
print(f'Original columns: {df.shape[1]}')
print(f'BERT embedding columns added: 768')
print(f'Total columns: {df_final.shape[1]}')

---
## Phase 2.5 Summary

In [ ]:
print('=== Phase 2.5 Complete ===')
print()
print('Model used        : bert-base-uncased (Google, 2018)')
print('Pre-trained on    : Wikipedia + BooksCorpus (3.3 billion words)')
print('Embedding size    : 768 dimensions per review')
print('Approach          : Feature extraction (BERT weights FROZEN)')
print('Transfer learning : Yes')
print('Data leakage risk : No (BERT is frozen - safe to run on full dataset)')
print()
print('Output files:')
print('  bert_embeddings.npy        - raw embeddings matrix (fast reload)')
print('  drug_reviews_phase2_5.csv  - full dataset ready for Phase 3')
print()
print('Features going into Phase 3:')
print('  From Phase 2  : sentiment_score, emotional_intensity, review_length,')
print('                  rating_tone_mismatch, tfidf_specificity,')
print('                  condition_category, useful_binary')
print('  From Phase 2.5: bert_1 ... bert_768 (BERT embeddings)')